# M5 Validation: Handcrafted Plan + LIBERO-Object + Pi0.5 VLA (Linux/Colab)

End-to-end M5 acceptance on a Google Colab Linux+CUDA VM. Runs the handcrafted plan executor on LIBERO-Object task 0 via the Pi0.5 VLA.

## 1 — Secrets & clone

In [ ]:
import getpass, os, shutil, subprocess, sys

github_repo = os.environ.get('GITHUB_REPO', 'pmish11/Longor-Robotics')

os.chdir('/')
print(os.getcwd())

if os.path.exists('/content/Longor'):
    shutil.rmtree('/content/Longor')

os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

result = subprocess.run(
    ['git', 'clone',
     f"https://{os.environ['GITHUB_TOKEN']}@github.com/{github_repo}.git",
     '/content/Longor'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr.replace(os.environ['GITHUB_TOKEN'], '***'))
    raise RuntimeError('git clone failed')

os.chdir('/content/Longor')
sys.path.insert(0, '/content/Longor/src')
print('Cloned OK')

## 2 — Headless rendering + app deps + LIBERO

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

# egl_probe / hf-egl-probe (pulled in by lerobot[libero]) need EGL headers and cmake to build from source
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r requirements-libero.txt  # includes -r requirements.txt

## 3 — VLA venv + Pi0.5 checkpoint

In [ ]:
import glob, os, shutil, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

# Colab (Debian/Ubuntu) doesn't ship python3-venv; virtualenv works without OS packages.
%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')  # pip self-upgrade can't go in a requirements file

if not os.path.exists('/root/openpi'):
    !git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git /root/openpi
else:
    print('/root/openpi already exists, skipping.')
vla_pip('install', '-e', '/root/openpi')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-pi05-linux.txt')

CKPT_CACHE = '/root/.cache/openpi/openpi-assets/checkpoints'
LIBERO_CKPT = os.path.join(CKPT_CACHE, 'pi05_libero')

for partial in glob.glob(f'{CKPT_CACHE}/*.partial'):
    shutil.rmtree(partial)
    print(f'Cleared partial: {partial}')

if os.path.isdir(LIBERO_CKPT):
    print(f'Checkpoint already present: {LIBERO_CKPT}')
else:
    os.makedirs(CKPT_CACHE, exist_ok=True)
    # crcmod C extension required for GCS composite-object integrity checks
    %pip install --quiet crcmod
    print('Downloading pi05_libero checkpoint via gsutil...')
    !gsutil -m -o GSUtil:num_retries=5 cp -r gs://openpi-assets/checkpoints/pi05_libero {CKPT_CACHE}
    print('Download complete.')

## 4 — Environment config

In [ ]:
import importlib.util, os, yaml

env_content = f"""OPENAI_API_KEY={os.environ['OPENAI_API_KEY']}
VLA_POLICY=pi05
VLA_MAX_TOOL_SECONDS=180
OPENPI_CONFIG=pi05_libero
BACKEND=libero
LLM_MODEL=gpt-4.1-mini
VLA_PORT=8765
"""
with open('/content/Longor/.env', 'w') as f:
    f.write(env_content)
print('Wrote /content/Longor/.env')

# libero/__init__.py calls input() if ~/.libero/config.yaml is absent; pre-create it to run non-interactively
libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')

if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

## 5 — Start VLA server

In [ ]:
import glob, json, os, shutil, subprocess, time, urllib.request

VLA_LOG = '/tmp/vla_pi05_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)  # ignore failure if port free

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

for partial in glob.glob('/root/.cache/openpi/openpi-assets/checkpoints/*.partial'):
    shutil.rmtree(partial)
    print(f'Cleared partial: {partial}')

# Pi0.5 blocks /health until checkpoint download + model load (GCS on first run can be slow).
log = open(VLA_LOG, 'w', buffering=1)
env_for_server = {**os.environ, 'OPENPI_CONFIG': 'pi05_libero'}
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server,
)

POLL_SEC = 15
MAX_WAIT_SEC = 20 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC

    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')

    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('VLA server ready:', json.loads(resp.read()))
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 6 — Run handcrafted example

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_log_path = f'{trace_dir}/m5_handcrafted_{stamp}.log'

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_libero_integration.py::test_handcrafted_libero_pick', '-m', 'integration', '-v'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={
        **os.environ,
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
        'LONGOR_RUN_LOG': run_log_path,
    },
)
print(result.stdout[-4000:])
print(result.stderr[-1500:])
print('Exit code:', result.returncode)
print(f'Run log: {run_log_path}')
combined = result.stdout + result.stderr
assert result.returncode == 0, f'exit code {result.returncode}'
assert 'ImportError' not in combined, 'Import failure detected'
assert 'ModuleNotFoundError' not in combined, 'Module not found'
assert 'Executor result:' in combined, 'Expected executor result in output'

## 7 — pytest + teardown

In [ ]:
import os, subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'},
)
print(result.stdout[-4000:])
print(result.stderr[-1000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

vla_proc.terminate()
print('VLA server stopped.')
